# Airline bookings: from messy CSV to dashboard
Made-up data. Question: which routes lose the most money to cancellations?

In [ ]:
import pandas as pd
df=pd.read_csv("airline_bookings_dirty.csv")
print("rows",len(df),"dup ids",df["Booking_ID"].duplicated().sum(),"null price %",round(df["Ticket_Price"].isna().mean()*100,1))
print("status values",df["Status"].nunique())
df.columns=df.columns.str.lower().str.strip()
df["status"]=df["status"].str.lower().str.strip()
df=df.drop_duplicates(subset="booking_id")
df["ticket_price"]=df["ticket_price"].fillna(df.groupby("route")["ticket_price"].transform("median"))
iso=pd.to_datetime(df["booking_date"],format="%Y-%m-%d",errors="coerce")
uk=pd.to_datetime(df["booking_date"],format="%d/%m/%Y",errors="coerce")
print("iso",iso.notna().sum(),"uk",uk.notna().sum())
df["booking_date"]=iso.fillna(uk)
print("bad dates",df["booking_date"].isna().sum())
df["is_cancelled"]=df["status"]=="cancelled"
df["month"]=df["booking_date"].dt.to_period("M").astype(str)
print("clean rows",len(df),"status",df["status"].unique(),"cancel %",round(df["is_cancelled"].mean()*100,1))
lost=df[df.is_cancelled].groupby("route")["ticket_price"].sum().sort_values(ascending=False)
print(lost.round(0)); print(df.groupby("route")["is_cancelled"].mean().round(3).sort_values(ascending=False)); print("total lost",round(lost.sum()), "bookings",df.groupby("month").size().to_dict())
df.to_csv("airline_bookings_clean.csv",index=False)


In [ ]:
import pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
df=pd.read_csv("airline_bookings_clean.csv")
rate=df.is_cancelled.mean()*100; lost=df[df.is_cancelled].ticket_price.sum()
by=df[df.is_cancelled].groupby("route").ticket_price.sum().sort_values()
mon=df.groupby("month").size()
f=make_subplots(rows=1,cols=2,subplot_titles=("Revenue lost to cancellations, by route (£)","Bookings per month"),horizontal_spacing=.12)
f.add_bar(y=by.index,x=by.values,orientation="h",marker_color=["#e0413a" if r=="LHR-SIN" else "#0a7cf0" for r in by.index],row=1,col=1)
f.add_scatter(x=mon.index,y=mon.values,mode="lines+markers",line=dict(color="#0a7cf0",width=3),row=1,col=2)
f.update_layout(title=dict(text=f"<b>Airline bookings</b>   |   {len(df):,} bookings   |   {rate:.1f}% cancelled   |   £{lost:,.0f} lost",font=dict(size=24)),showlegend=False,template="plotly_white",width=1500,height=720,font=dict(family="Inter, Arial",size=17),margin=dict(l=90,r=40,t=120,b=70))
f.update_annotations(font_size=19); f.update_yaxes(rangemode="tozero",row=1,col=2)
f.write_html("dashboard.html",include_plotlyjs=True,full_html=True)
